#산불 데이터 입력

###산불 이력 데이터 정제


In [ ]:
from google.colab import drive

# Google Drive 마운트
drive.mount('/content/drive')
# CSV 불러오기
fire = pd.read_csv("/content/국내산불정보(재난안전데이터공유플랫폼) /wildfire_raw_page1_to_167.csv")

print(fire.head())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
   FORST_PSN_SE_CD  FRST_RGTR_ID FTRXTNGSH_SRNC_END_DT  \
0              NaN           NaN                   NaN   
1              NaN           NaN                   NaN   
2              NaN           NaN                   NaN   
3              NaN           NaN                   NaN   
4              NaN           NaN                   NaN   

               FRSTFR_DCLR_ADDR GRNDS_0_RBPRSN_NM FRSTFR_DCL_NM  \
0        경상북도 문경시 산양면 반곡리 95-2임               NaN           NaN   
1      경상남도 고성군 하이면 월흥리 1212-1대               NaN           NaN   
2  충청북도 청주시 청원구 내수읍 은곡리 170-3 대               NaN           NaN   
3        경기도 고양시 덕양구 대자동 255-1장               NaN           NaN   
4          경상남도 합천군 야로면 야로리 산5임               NaN           NaN   

   FRSTFR_DCLR_YMD                                LINK_TRNS_ID  \
0              NaN  IF_KFS_MPSS_001_20250801193159001

In [16]:
fire.info() #https://www.safetydata.go.kr/disaster-data/view?dataSn=882

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16638 entries, 0 to 16637
Data columns (total 41 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   FORST_PSN_SE_CD         9 non-null      float64
 1   FRST_RGTR_ID            0 non-null      float64
 2   FTRXTNGSH_SRNC_END_DT   9 non-null      object 
 3   FRSTFR_DCLR_ADDR        16638 non-null  object 
 4   GRNDS_0_RBPRSN_NM       18 non-null     object 
 5   FRSTFR_DCL_NM           29 non-null     object 
 6   FRSTFR_DCLR_YMD         0 non-null      float64
 7   LINK_TRNS_ID            16638 non-null  object 
 8   FRSTFR_OCRN_CS_DTL_CN   0 non-null      float64
 9   LINK_STTS               16638 non-null  object 
 10  FRSTFR_PSTN_XCRD        16638 non-null  float64
 11  FRSTFR_OCRN_CS_CLSF_CD  9 non-null      float64
 12  FRST_MDFCN_DT           16638 non-null  object 
 13  FRSTFR_GNT_DT           16638 non-null  object 
 14  FRSTFR_GRS_DAM_AMT      9 non-null    

In [17]:
fire.isnull().sum()  #Total rows : 16638
#  필수 정보 : FRSTFR_PSTN_XCRD (X좌표) , 	FRSTFR_PSTN_YCRD (Y좌표) , 	FRSTFR_DCLR_ADDR (산불 신고 장소), FRSTFR_GNT_DT 산불발생일자

,0
FORST_PSN_SE_CD,16629
FRST_RGTR_ID,16638
FTRXTNGSH_SRNC_END_DT,16629
FRSTFR_DCLR_ADDR,0
GRNDS_0_RBPRSN_NM,16620
FRSTFR_DCL_NM,16609
FRSTFR_DCLR_YMD,16638
LINK_TRNS_ID,0
FRSTFR_OCRN_CS_DTL_CN,16638
LINK_STTS,0


In [18]:
#  같은 좌표가 있는지 체크 / 그 근방에서 산불 횟수가 많을 수록 크게 표시
fire[["FRSTFR_PSTN_YCRD", "FRSTFR_PSTN_XCRD"]].value_counts()


,,count
FRSTFR_PSTN_YCRD,FRSTFR_PSTN_XCRD,
36.575990,128.505800,18
35.003475,128.064956,12
33.392862,126.494438,9
35.002881,128.065597,9
35.003642,128.064626,8
...,...,...
35.864531,127.630770,1
35.864844,128.446219,1
35.865010,128.867200,1


In [19]:
fire["FRSTFR_DCLR_ADDR"].value_counts()

,count
FRSTFR_DCLR_ADDR,
경상남도 사천시 용현면 덕곡리 501 대,83
대전광역시 서구 둔산동 920대,76
경상남도 사천시 곤명면 봉계리 990-8 대,50
부산광역시 강서구 명지동 3247공,27
경상북도 안동시 풍천면 갈전리 1155 대,21
...,...
경상북도 예천군 감천면 현내리 916,1
경기도 고양시 덕양구 대자동 255-1장,1
세종특별자치시 부강면 부강리 산37임,1


In [ ]:
# ==========================
# WRI 계산 + 영상 팝업 + 산불 이력 + 물모이 상태 분리까지 업데이트 
# ==========================

# ---------- Imports ----------
import math
import pandas as pd
import numpy as np
import folium
from folium.plugins import Search
from folium import Element
from folium.features import DivIcon

# ---------- 0. 유틸 ----------
def haversine_km(lat1, lon1, lat2, lon2):
    """두 좌표 사이 하버사인 거리(km)"""
    R = 6371.0088
    dlat = math.radians(lat2 - lat1)
    dlon = math.radians(lon2 - lon1)
    a = (math.sin(dlat/2)**2
         + math.cos(math.radians(lat1))*math.cos(math.radians(lat2))*math.sin(dlon/2)**2)
    return 2 * R * math.asin(math.sqrt(a))

def winsorize(s, q=0.01):
    """극값 안정화(윈저라이즈)"""
    s = pd.to_numeric(s, errors="coerce")
    lo, hi = s.quantile(q), s.quantile(1 - q)
    return s.clip(lo, hi)

def norm01(series, reverse=False):
    """0~1 정규화 (reverse=True면 역방향)"""
    s = winsorize(series.astype(float))
    v = (s - s.min()) / (s.max() - s.min() + 1e-9)
    return 1 - v if reverse else v

def classify_risk(x):
    if x >= 0.75: return "매우 높음"
    if x >= 0.50: return "높음"
    if x >= 0.25: return "보통"
    return "낮음"

# ---------- 1. 산불 발생 이력 (최근 100건만) ----------
try:
    cols = ['FRSTFR_PSTN_XCRD', 'FRSTFR_PSTN_YCRD', 'FRSTFR_DCLR_ADDR', 'FRSTFR_GNT_DT']
    wildfire_data = (fire[cols].copy()
                    .rename(columns={
                        'FRSTFR_PSTN_XCRD': 'lon',
                        'FRSTFR_PSTN_YCRD': 'lat',
                        'FRSTFR_DCLR_ADDR': 'name',
                        'FRSTFR_GNT_DT'   : 'date'
                    }))
    wildfire_data['date'] = pd.to_datetime(wildfire_data['date'])
    wildfire_recent = wildfire_data.sort_values('date', ascending=False).head(100)
    agg = (wildfire_recent
           .groupby(['lat', 'lon', 'name'], as_index=False)
           .agg(count=('name', 'size'),
                date=('date', 'max')))
    df_fire = agg[['name', 'lat', 'lon', 'date', 'count']]
except Exception:
    # fire 변수가 없거나 컬럼이 다르면 빈 프레임
    df_fire = pd.DataFrame(columns=['name','lat','lon','date','count'])

# ---------- 2. 물모이 설치 지점(상태 분리 가능) ----------
# status: 'installed' (설치), 'needed' (설치 필요)
water_storage = [
    {"name": "강원특별자치도 정선군 북평면 문곡리 121", "lat": 37.421336, "lon": 128.699194, "status": "installed"},
    {"name": "전북특별자치도 진안군 정천면 봉학리", "lat": 35.877779, "lon": 127.399781, "status": "installed"},
    {"name": "강원특별자치도 강릉시 구정 제비리", "lat": 37.716714, "lon": 128.848508, "status": "installed"},

    {"name": "강원특별자치도 태백시 상장동", "lat": 37.163746, "lon": 128.967937, "status": "installed"},
    {"name": "전라남도 화순군 사평면 사수리", "lat": 34.984371, "lon": 127.128038, "status": "installed"},

    {"name": "충청남도 부여군 충화면 천당리", "lat": 36.169740, "lon": 126.815124, "status": "installed"},

    {"name": "경상북도 문경시 호계면 별암리", "lat": 36.660753, "lon": 128.165926, "status": "needed"},
    {"name": "경상남도 양산시 원동면 선리", "lat": 35.486424, "lon": 128.975680, "status": "needed"},

    {"name": "경상남도 합천군 가야면 죽전리", "lat": 35.765463, "lon": 128.055766, "status": "needed"},
    {"name": "전라남도 구례군 토지면 문수리", "lat": 35.280408,  "lon":127.535118, "status": "needed"},
]
df_water = pd.DataFrame(water_storage)


#!!!!!!!!!!!!!!!!!!!!!soil_pred --> 날씨 api 연동하여 예측 필요 !!!!!!!!!!!!!!!

# ---------- 3. 관측/예측 지점 입력  ----------
# - road_dist_m 는 자동 계산(아래에서 nearest_road_lat/lon 으로 하버사인 dist 계산)
# - water_dist_m: 이미 설치된 물모이까지의 거리. 설치 전이면 -1로 넣으면 WRI에서 자동 제외
stations = [
    # ==== 물모이 설치 지점 6개 ====
    #37.416415, 128.697102 / 문곡강변길,강원특별자치도 정선군 북평면
    {
        "name": "강원특별자치도 정선군 북평면 문곡리 산54", "lat": 37.42130, "lon": 128.7009,
        "nearest_road_lat": 37.416415, "nearest_road_lon": 128.697102,
        "soil_pred": 28.5, "slope": 8.75, "water_dist_m": 134.53
    },
    #35.882430, 127.405782 / 봉학로,전북특별자치도 진안군 정천면
    {
        "name": "전북특별자치도 진안군 정천면 봉학리", "lat": 35.876595, "lon": 127.397878,
        "nearest_road_lat": 35.882430, "nearest_road_lon": 127.405782,
        "soil_pred": 35.7, "slope": 70.72, "water_dist_m": 207.59
    },
    #37.717362, 128.856214 / 강원특별자치도 강릉시 구정면 제비리 564
    {
        "name": "강원특별자치도 강릉시 구정면 제비리", "lat": 37.716812, "lon": 128.848199,
        "nearest_road_lat": 37.7525, "nearest_road_lon": 128.8700,
        "soil_pred": 22.4, "slope": 70.02, "water_dist_m": 29.32
    },
    #37.166158, 128.971740 / 강원특별자치도 태백시 황지동 484
    {
        "name": "강원특별자치도 태백시 상장동", "lat": 37.164691, "lon": 128.968055,
        "nearest_road_lat": 37.166158, "nearest_road_lon": 128.971740,
        "soil_pred": 20.5, "slope": 57.74, "water_dist_m": 105.85
    },
    #34.988692, 127.128218 / 전라남도 화순군 사평면 사수리 963
    {
        "name": "전라남도 화순군 사평면 사수리", "lat": 34.984942, "lon":127.128060,
        "nearest_road_lat": 34.988692, "nearest_road_lon": 127.128218,
        "soil_pred": 26.8, "slope": 70.02, "water_dist_m": 64.75
    },
    #36.172482, 126.817935 / 천당리,충청남도 부여군 충화면
    {
        "name": "충청남도 부여군 충화면 청남리 산149", "lat": 36.170132, "lon": 126.815365,
        "nearest_road_lat": 36.172482, "nearest_road_lon": 126.817935,
        "soil_pred": 24.6, "slope": 57.74, "water_dist_m": 47.90
    },

    # ==== 물모이 미설치 지점 4개 ====
    #36.658841, 128.173174 / 경상북도 문경시 호계면 별암리 730
    {
        "name": "경상북도 문경시 호계면 별암리", "lat": 36.661327, "lon": 128.165853,
        "nearest_road_lat": 36.658841, "nearest_road_lon": 128.173174,
        "soil_pred": 31.0, "slope": 46.63, "water_dist_m": -1    # 미설치
    },
    #35.486301, 128.983603 / 경상남도 양산시 원동면 선리 1614
    {
        "name": "경상남도 양산시 원동면 선리", "lat": 35.487365, "lon": 128.976630,
        "nearest_road_lat": 35.48630, "nearest_road_lon": 128.983603,
        "soil_pred": 27.5, "slope": 57.74, "water_dist_m": -1    # 미설치
    },
    #35.762145, 128.070405 / 경상남도 합천군 가야면 죽전리 138
    {
        "name": "경상남도 합천군 가야면 죽전리", "lat": 35.766554, "lon": 128.055884,
        "nearest_road_lat": 35.762145, "nearest_road_lon": 128.070405,
        "soil_pred": 33.8, "slope": 70.02, "water_dist_m": -1 # 미설치
    },
    #35.264490, 127.533164 / 전라남도 구례군 토지면 문수리 236
    {
        "name": "전라남도 구례군 토지면 문수리", "lat": 35.281048, "lon": 127.535151,
        "nearest_road_lat": 35.264490, "nearest_road_lon": 127.533164,
        "soil_pred": 29.2, "slope": 46.63, "water_dist_m": -1 # 미설치
    }
]

df = pd.DataFrame(stations)

# ---------- 4. 도로까지 거리(m) 계산 (임의로 넣은 최근접 도로 좌표 사용) ----------
def compute_road_dist_from_manual(df):
    vals = []
    for r in df.itertuples():
        if ("nearest_road_lat" in df.columns and "nearest_road_lon" in df.columns and
            pd.notna(r.nearest_road_lat) and pd.notna(r.nearest_road_lon)):
            d_m = haversine_km(r.lat, r.lon, r.nearest_road_lat, r.nearest_road_lon) * 1000.0
        else:
            d_m = np.nan
        vals.append(d_m)
    return vals

df["road_dist_m"] = compute_road_dist_from_manual(df)

# ---------- 5. (선택) water_dist_m 미입력(-1)일 때 설치된 물모이까지의 최단거리 자동 보완 ----------
def nearest_installed_water(lat, lon, df_water):
    w_inst = df_water[df_water["status"] == "installed"]
    if w_inst.empty:
        return np.nan, np.nan, np.nan
    dmin = np.inf; best = None
    for _, w in w_inst.iterrows():
        d = haversine_km(lat, lon, w.lat, w.lon)*1000.0
        if d < dmin:
            dmin = d; best = (w.lat, w.lon)
    if best is None:
        return np.nan, np.nan, np.nan
    return dmin, best[0], best[1]

if "water_dist_m" not in df.columns:
    df["water_dist_m"] = -1  # 전부 미설치로 간주

linked_lat = []
linked_lon = []
final_water_d = []

for r in df.itertuples():
    # 사용자가 -1을 넣은 경우(미설치) → 설치된 물모이가 있으면 그 최단거리로 보완, 없으면 -1 유지
    if getattr(r, "water_dist_m", -1) == -1:
        d, la, lo = nearest_installed_water(r.lat, r.lon, df_water)
        if np.isnan(d):
            final_water_d.append(-1)
            linked_lat.append(np.nan)
            linked_lon.append(np.nan)
        else:
            final_water_d.append(d)
            linked_lat.append(la)
            linked_lon.append(lo)
    else:
        # 사용자가 거리값을 지정한 경우: 표시용 연결선은 '가장 가까운 설치 물모이'로 연결(없으면 연결 X)
        d, la, lo = nearest_installed_water(r.lat, r.lon, df_water)
        if np.isnan(d):
            linked_lat.append(np.nan)
            linked_lon.append(np.nan)
        else:
            linked_lat.append(la)
            linked_lon.append(lo)
        final_water_d.append(r.water_dist_m)

df["water_dist_m"] = final_water_d
df["linked_water_lat"] = linked_lat
df["linked_water_lon"] = linked_lon


# ---------- 6. WRI 계산 ----------
# 6-1) 정규화 대상/방향
# soil: 역방향(클수록 위험↓), slope/road/water: 정방향(클수록 위험↑)
spec = {
    "soil_pred":   {"reverse": True,  "alias": "soil"},
    "slope":       {"reverse": False, "alias": "slope"},
    "road_dist_m": {"reverse": False, "alias": "road"},
    "water_dist_m":{"reverse": False, "alias": "water"},
}

# 6-2) 기본 가중치
base_weights = {
    "soil": 0.35,
    "slope":0.20,
    "road": 0.20,
    "water":0.25,
}

# 6-3) water_dist_m == -1 인 지점은 'water' 변수를 WRI에서 제외
#     => 계산을 위해 해당 지점만 NaN으로 바꾸고, 가중치를 재정규화
df_calc = df.copy()

# water_dist_m == -1 -> NaN 처리(그 지점에 한해 제외)
mask_exclude_water = (df_calc["water_dist_m"] == -1)
df_calc.loc[mask_exclude_water, "water_dist_m"] = np.nan

# 6-4) 각 변수 정규화 컬럼 생성
norm_cols = {}
for raw_col, cfg in spec.items():
    if raw_col in df_calc.columns:
        alias = cfg["alias"]
        df_calc[f"{alias}_norm"] = norm01(df_calc[raw_col], reverse=cfg["reverse"])
        norm_cols[alias] = f"{alias}_norm"

# 6-5) 지점별로 '사용 가능한 변수'만 가중치 재정규화 후 합산
def compute_wri_row(row, norm_cols, base_weights):
    # 사용 가능한(alias: 값 존재) 변수만
    avail = []
    for alias, col in norm_cols.items():
        val = row.get(col, np.nan)
        if pd.notna(val) and alias in base_weights:
            avail.append(alias)
    if not avail:
        return np.nan
    # 가중치 재정규화
    w = np.array([base_weights[a] for a in avail], dtype=float)
    w = w / (w.sum() + 1e-12)
    # 가중 합
    vals = np.array([row[norm_cols[a]] for a in avail], dtype=float)
    return float((w * vals).sum())

df_calc["WRI"] = df_calc.apply(lambda r: compute_wri_row(r, norm_cols, base_weights), axis=1)
df_calc["risk_level"] = df_calc["WRI"].apply(classify_risk)

# 표시용 df 갱신
df = df_calc.copy()

# ---------- (설치 필요 후보에 대한 WRI 개선 시뮬레이션 함수) ----------
def wri_with_water_override(df_all, row_idx, new_water_dist_m):
    """
    특정 지점(row_idx)에 water_dist_m을 new_water_dist_m로 가정했을 때의 WRI.
    정규화는 전체 df 기준으로 다시 계산(해당 지점의 가상값 포함).
    """
    tmp = df_all.copy()
    tmp.loc[row_idx, "water_dist_m"] = new_water_dist_m
    tmp2, ncols = compute_norm_cols(tmp.copy(), spec)

    def one_row_wri(r):
        avail = []
        for alias, col in ncols.items():
            if pd.notna(r.get(col, np.nan)) and alias in base_weights:
                avail.append(alias)
        if not avail:
            return np.nan
        w = np.array([base_weights[a] for a in avail], float); w = w/(w.sum()+1e-12)
        vals = np.array([r[ncols[a]] for a in avail], float)
        return float((w*vals).sum())

    return one_row_wri(tmp2.loc[row_idx])


# ---------- 7. 지도 구성 ----------
center = [df['lat'].mean(), df['lon'].mean()]
m = folium.Map(location=center, zoom_start=9, tiles='openstreetmap')

# 검색창 스타일
style = """
<style>
.leaflet-control-search input {
    width: 260px !important;
    font-size: 14px !important;
}
.search-only { display: none !important; }
</style>
"""
m.get_root().html.add_child(Element(style))

# 색상 맵
#'cadetblue', 'darkpurple', 'pink', 'blue', 'lightgreen', 'darkgreen', 'darkred', 'lightgray', 'green', 'purple', 'beige', 'darkblue', 'orange', 'lightred', 'red', 'gray', 'lightblue', 'black', 'white'
color_map = {
    "낮음":     "green",
    "보통":     "beige",
    "높음":     "orange",
    "매우 높음": "red",
}


# 레이어
risk_fg          = folium.FeatureGroup(name='산불 발생 위험 지역').add_to(m)
water_inst_fg    = folium.FeatureGroup(name='물모이 설치').add_to(m)
water_needed_fg  = folium.FeatureGroup(name='물모이 설치 필요').add_to(m)
fire_fg          = folium.FeatureGroup(name='산불 발생 이력').add_to(m)
search_fg        = folium.FeatureGroup(name='(검색전용)', control=False).add_to(m)

# ---------- 8. 위험지수 마커 + 검색 전용 ----------
# (선택) 영상 팝업 매핑
video_map = {
    "전라남도 구례군 토지면 문수리": "https://www.youtube.com/embed/TWAU8hGF2qs?si=v3nRICrMDGwbxfu5",
    "강원특별자치도 정선군 북평면 문곡리 산54": "https://www.youtube.com/embed/4ofdxht-jLY?si=mXmOUhnGlqxhqvAG",
}
detect_text = {
    "전라남도 구례군 토지면 문수리": {"msg": "AI 감지: 연기 패턴 포착", "conf": 0.92, "time": "2025-09-15 14:32"},
    "강원특별자치도 정선군 북평면 문곡리 산54": {"msg": "AI 감지: 열원 이상치",     "conf": 0.87, "time": "2025-09-15 14:35"},
}

for _, row in df.iterrows():
    name = row['name']
    video_url = video_map.get(name)
    info = detect_text.get(name)

    msg  = info['msg'] if info else "AI 감지 정보"
    conf = f"{info['conf']*100:.0f}%" if info and 'conf' in info else "-"
    time = info['time'] if info and 'time' in info else "-"
    popup_html = f"""
    <div style="width:360px">
        <div style="font-weight:600;margin-bottom:6px">{name}</div>
        <iframe width="360" height="315" src="{video_url}" title="YouTube video player" frameborder="0" allow="accelerometer; autoplay; clipboard-write; encrypted-media; gyroscope; picture-in-picture; web-share" referrerpolicy="strict-origin-when-cross-origin" allowfullscreen></iframe>
        <div style="margin-top:6px; line-height:1.4">
        <div>WRI: {row['WRI']:.2f} · 등급: {row['risk_level']}</div>
        <div>{msg} · Confidence: {conf}</div>
        <div>Time: {time}</div>
        <div>도로거리: {row.get('road_dist_m', np.nan):.0f} m</div>
        <div>물모이거리: {"미설치" if (pd.isna(row.get("water_dist_m")) or row.get("water_dist_m") == -1) else f"{row.get('water_dist_m'):.0f} m"}</div>
        </div>
    </div>
    """
    popup_obj = folium.Popup(popup_html, max_width=420)

    folium.Marker(
        location=(row['lat'], row['lon']),
        tooltip=name,
        icon=folium.Icon(color=color_map[row['risk_level']], icon='fire', prefix='fa'),
        popup=popup_obj
    ).add_to(risk_fg)

    # 검색 전용 (화면엔 숨김)
    folium.Marker(
        location=(row['lat'], row['lon']),
        title=row['name'],
        icon=DivIcon(html="", class_name="search-only")
    ).add_to(search_fg)

# ---------- 9. 물모이 마커: 설치 / 설치 필요 ----------
# (2) '설치 필요' 팝업에 설치시 WRI 변화 시뮬 표시
NEARBY_KM = 10  # 이 물모이가 커버한다고 보는 반경 (필요 시 조정)
TOPK = 3        # 상위 몇 개 지점 표시

for _, row in df_water.iterrows():
    icon_color = 'cadetblue' if row['status'] == 'installed' else 'lightgray'
    layer = water_inst_fg if row['status'] == 'installed' else water_needed_fg

    # 기본 툴팁/팝업
    title = f"{row['name']} ({'설치' if row['status']=='installed' else '설치 필요'})"

    if row['status'] == 'needed':
        # 가까운 지점들에 대해 '이 물모이를 설치했다면'의 가상 water_dist로 WRI_new 계산
        sim_rows = []
        for idx, s in df.iterrows():
            d_m = haversine_km(s.lat, s.lon, row.lat, row.lon)*1000.0
            if d_m <= NEARBY_KM*1000:  # 반경 내만 고려
                wri_old = s.WRI
                wri_new = wri_with_water_override(df, idx, d_m)
                if pd.notna(wri_new) and pd.notna(wri_old):
                    sim_rows.append({
                        "name": s.name, "dist_m": d_m,
                        "old": wri_old, "new": wri_new, "delta": wri_new - wri_old
                    })
        #sim_df = pd.DataFrame(sim_rows).sort_values("delta").head(TOPK)  # ΔWRI가 작을수록(위험↓) 상위

        if len(sim_rows) == 0:
            sim_html = "<div style='color:#999'>반경 내 연결 가능한 지점이 없습니다.</div>"
        else:
            sim_df = pd.DataFrame(sim_rows)
            # delta가 NaN인 행 제거 후 정렬
            if "delta" in sim_df.columns:
                sim_df = sim_df.dropna(subset=["delta"]).sort_values("delta", na_position="last").head(TOPK)
            else:
                sim_df = pd.DataFrame(columns=["name","dist_m","old","new","delta"])  # 안전 가드

            if sim_df.empty:
                sim_html = "<div style='color:#999'>반경 내 연결 가능한 지점이 없습니다.</div>"
            else:
                items = []
                # itertuples 대신 iterrows 사용(인덱스/필드명 혼동 방지)
                for _, r2 in sim_df.iterrows():
                    items.append(
                        f"<div style='display:flex;justify-content:space-between;'>"
                        f"<span>{r2['name']} · {r2['dist_m']:.0f}m</span>"
                        f"<span>WRI {r2['old']:.2f} → {r2['new']:.2f} "
                        f"<b style='color:#2E86C1'>(Δ{r2['delta']:+.2f})</b></span>"
                        f"</div>"
                    )
                sim_html = "<br>".join(items)



        popup_html = f"""
        <div style="width:360px; padding:6px; font-size:13px;">
            <b style="font-size:14px;">{title}</b>
            <hr style="margin:6px 0;">
            <div style="margin-bottom:6px"><b>설치 시 예상 WRI 개선 (반경 {NEARBY_KM}km)</b></div>
            {sim_html}
        </div>
        """
    else:
        popup_html = f"""
        <div style="width:280px; padding:6px; font-size:13px;">
            <b style="font-size:14px;">{title}</b>
            <div style="margin-top:4px; color:#666">설치 완료 지점</div>
        </div>
        """

    folium.Marker(
        location=(row['lat'], row['lon']),
        tooltip=title,
        icon=folium.Icon(color=icon_color, icon='tint', prefix='fa'),
        popup=folium.Popup(popup_html, max_width=420)
    ).add_to(layer)

    # 검색 전용(숨김)
    folium.Marker(
        location=(row['lat'], row['lon']),
        title=title,
        icon=DivIcon(html="", class_name="search-only")
    ).add_to(search_fg)

# ---------- 10. 산불 이력 마커 ----------
for _, row in df_fire.iterrows():
    name_fire = f"{row['name']}"
    # 팝업 카드
    if pd.notna(row.get('date', pd.NaT)):
        date_str = pd.to_datetime(row['date']).strftime('%Y-%m-%d %H:%M:%S')
    else:
        date_str = "-"

    popup_html = f"""
    <div style="width:360px; padding:6px; font-size:13px;">
        <b style="font-size:14px; color:#E74C3C;">{name_fire}</b>
        <hr style="margin:4px 0;">
        <div style="display:flex; justify-content:space-between;"><span><b>발생일:</b> {date_str}</span></div>
        <div style="display:flex; justify-content:space-between;"><span><b>발생 횟수:</b> {row.get('count','-')}</span></div>
        <div style="display:flex; justify-content:space-between;"><span><b>좌표:</b> {row['lat']:.4f}, {row['lon']:.4f}</span></div>
    </div>
    """
    popup_obj = folium.Popup(popup_html, max_width=420)

    folium.CircleMarker(
        location=(row['lat'], row['lon']),
        radius=5,
        color= "#E74C3C",
        fill_color= "#E67E22",
        fill_opacity=0.85,
        popup=popup_obj
    ).add_to(fire_fg)

    '''folium.Marker(
        location=(row['lat'], row['lon']),
        tooltip=name_fire,
        icon=folium.Icon(color='red', icon='fire', prefix='fa'),
        popup=popup_obj
    ).add_to(fire_fg)'''

    # 검색 전용
    folium.Marker(
        location=(row['lat'], row['lon']),
        title=name_fire,
        icon=DivIcon(html="", class_name="search-only")
    ).add_to(search_fg)


# ---------- 11. 설치된 물모이에 직선 연결 ----------
# 각 지점이 참조하고 있는 linked_water_lat/lon 으로 선을 그림
for r in df.itertuples():
    if pd.notna(r.linked_water_lat) and pd.notna(r.linked_water_lon):
        folium.PolyLine(
            locations=[(r.lat, r.lon), (r.linked_water_lat, r.linked_water_lon)],
            color="#2E86C1", weight=2, opacity=0.7, dash_array="4,6"
        ).add_to(water_inst_fg)

# ---------- 11. Search / LayerControl ----------
Search(
    search_fg,
    search_label="title",
    placeholder='지점/물모이/산불 이름 검색',
    collapsed=False,
    position='topleft'
).add_to(m)

folium.LayerControl(collapsed=False, position='topright').add_to(m)

# ---------- 12. 출력/저장 ----------
from IPython.display import display
display(m)
m.save('ksef_wri_complete_map.html')

# ==========================
# 끝
# ==========================



#추가 자료 확보 중

#다시 학습해서 정확도 비교

In [11]:
df


,name,lat,lon,soil_pred,slope,road_dist,water_dist,soil_norm,slope_norm,road_norm,water_norm,WRI,risk_level
0,정선,37.3800,128.6620,28.5,12.3,1.5,0.8,1.000000e+00,1.0,0.0,0.0,0.55,높음
1,부귀,35.8623,127.3993,35.7,9.5,2.0,1.2,1.388889e-10,0.0,1.0,1.0,0.25,낮음


#백업


###FastMarkerCluster 사용 버전, 아이콘 개별 지정이 어렵고 검색 불가, 팝업 띄우기가 어려움 등으로 보류

In [ ]:
import pandas as pd
import numpy as np
import requests
import folium
from folium.plugins import Search
from folium.plugins import FastMarkerCluster
from folium import Element


stations = [
    {"name": "정선", "lat": 37.3800, "lon": 128.6620, "soil_pred": 28.5, "slope": 12.3, "road_dist": 1.5, "water_dist": 0.8},
    {"name": "부귀", "lat": 35.8623, "lon": 127.3993, "soil_pred": 35.7, "slope": 9.5, "road_dist": 2.0, "water_dist": 1.2},

]
df = pd.DataFrame(stations)

# 필요한 4개 컬럼만 추출 → 표준 컬럼명으로 변경
cols = ['FRSTFR_PSTN_XCRD', 'FRSTFR_PSTN_YCRD', 'FRSTFR_DCLR_ADDR', 'FRSTFR_GNT_DT']
wildfire_data = (fire[cols].copy()
              .rename(columns={'FRSTFR_PSTN_XCRD':'lon',
                               'FRSTFR_PSTN_YCRD':'lat',
                               'FRSTFR_DCLR_ADDR':'name',
                               'FRSTFR_GNT_DT' : 'date'}))

# 같은 좌표/주소 묶어서 개수 세기
agg = (wildfire_data
       .groupby(['lat','lon','name'], as_index=False)
       .agg(count=('name','size'),
            date=('date','max')))

# stations 형식으로 변환 (이름에 건수 표시)
df_fire = agg[['name','lat','lon','date','count']]

water_storage = [
    {"name": "물모이A", "lat": 37.40, "lon": 128.65},
    {"name": "물모이B", "lat": 35.88, "lon": 127.55},
]
df_water = pd.DataFrame(water_storage)


# 지점별(또는 카메라별) 영상 URL/경로 매핑 (예시)
video_map = {
    "정선": "files/jeongseon.mp4",  # 업로드한 파일명으로 교체
    "부귀": "files/bugwi.mp4",      # 업로드한 파일명으로 교체
    # 필요 시 추가
}
# 지점별 감지 설명(예시)
detect_text = {
    "정선": {"msg": "AI 감지: 연기 패턴 포착", "conf": 0.92, "time": "2025-09-15 14:32"},
    "부귀": {"msg": "AI 감지: 열원 이상치",     "conf": 0.87, "time": "2025-09-15 14:35"},
}


# ----------------------------------------------------------
# 2. 변수 정규화 및 WRI 계산
def minmax(series, reverse=False):
    s = (series - series.min()) / (series.max() - series.min() + 1e-9)
    return 1 - s if reverse else s

df['soil_norm']  = minmax(df['soil_pred'], reverse=True)
df['slope_norm'] = minmax(df['slope'])
df['road_norm']  = minmax(df['road_dist'])
df['water_norm'] = minmax(df['water_dist'])

weights = {
    'soil_norm': 0.4,
    'slope_norm': 0.15,
    'road_norm': 0.15,
    'water_norm': 0.1,
}
df['WRI'] = sum(df[col] * w for col, w in weights.items())

def classify_risk(wri):
    if wri >= 0.75: return "매우 높음"
    elif wri >= 0.5: return "높음"
    elif wri >= 0.25: return "보통"
    else: return "낮음"

df['risk_level'] = df['WRI'].apply(classify_risk)


# ----------------------------------------------------------
# 3-1. 지도 생성
center = [df['lat'].mean(), df['lon'].mean()]
m = folium.Map(location=center, zoom_start=9, tiles='openstreetmap')

# 검색창 폭/글꼴 크기 조정
style = """
<style>
.leaflet-control-search input {
    width: 260px !important;
    font-size: 14px !important;
}
.search-only { display: none !important; }
</style>
"""
m.get_root().html.add_child(Element(style))

# 위험 색상
color_map = {
    "낮음": "#2ECC71",
    "보통": "#F1C40F",
    "높음": "#E67E22",
    "매우 높음": "#E74C3C",
}

# ----------------------------------------------------------
# 3-2. 레이어(토글용)
risk_fg  = folium.FeatureGroup(name='산불 발생 위험 지역').add_to(m)
water_fg = folium.FeatureGroup(name='물모이 설치 지역').add_to(m)
fire_fg  = folium.FeatureGroup(name='산불 발생 이력').add_to(m)

# 검색 전용 레이어 (LayerControl에 안 보이게 control=False)
search_fg = folium.FeatureGroup(name='(검색전용)', control=False).add_to(m)

# ----------------------------------------------------------
# 3-3. 위험지수 마커(위험 지역) + 검색 전용 마커
for _, row in df.iterrows():
    name = row['name']
    video_url = video_map.get(name)
    info = detect_text.get(name)

    if video_url:
        # 영상 + 텍스트 팝업 (너비 320px 기준)
        msg = info['msg'] if info else "AI 감지 정보"
        conf = f"{info['conf']*100:.0f}%" if info and 'conf' in info else "-"
        time = info['time'] if info and 'time' in info else "-"
        popup_html = f"""
        <div style="width:320px">
          <div style="font-weight:600;margin-bottom:6px">{name}</div>
          <video width="320" controls autoplay muted playsinline>
            <source src="{video_url}" type="video/mp4">
            브라우저가 video 태그를 지원하지 않습니다.
          </video>
          <div style="margin-top:6px; line-height:1.4">
            <div>WRI: {row['WRI']:.2f} · 등급: {row['risk_level']}</div>
            <div>{msg} · Confidence: {conf}</div>
            <div>Time: {time}</div>
          </div>
        </div>
        """
        popup_obj = folium.Popup(popup_html, max_width=360)
    else:
        # 영상이 없으면 텍스트 팝업
        popup_obj = folium.Popup(
            f"{name}<br>WRI: {row['WRI']:.2f}<br>등급: {row['risk_level']}",
            max_width=250
        )

    folium.CircleMarker(
        location=(row['lat'], row['lon']),
        radius=8,
        color= color_map[row['risk_level']],
        fill_color= color_map[row['risk_level']],
        fill_opacity=0.85,
        popup=popup_obj
    ).add_to(risk_fg)

    # 검색 전용 Marker
    folium.Marker(
        location=(row['lat'], row['lon']),
        title=row['name'],
        icon=DivIcon(html="", class_name="search-only")
    ).add_to(search_fg)


# ----------------------------------------------------------
# 3-4. 물모이 마커 + 검색 전용 마커
for _, row in df_water.iterrows():
    folium.Marker(
        location=(row['lat'], row['lon']),
        tooltip=row['name'],
        icon=folium.Icon(color='cadetblue', icon='tint', prefix='fa'),
        popup=f"{row['name']}"
    ).add_to(water_fg)

    folium.Marker(
        location=(row['lat'], row['lon']),
        title=row['name'],
        icon=DivIcon(html="", class_name="search-only")
    ).add_to(search_fg)

# ----------------------------------------------------------
cluster = MarkerCluster(
    name='산불 발생 이력(클러스터)',
    options={'spiderfyOnMaxZoom': False, 'disableClusteringAtZoom': 12}
).add_to(m)

# 3-5. 산불 발생 이력 마커 + 검색 전용 마커
fire_cluster = FastMarkerCluster(
    data=df_fire[['lat','lon']].dropna().values.tolist(),
    name='산불 발생 이력(클러스터)'
).add_to(m)

# ----------------------------------------------------------
# 3-6. Search 플러그인
Search(
    search_fg,
    search_label="title",
    placeholder='지점/물모이/산불 이름 검색',
    collapsed=False,
    position='topleft'
).add_to(m)

# ----------------------------------------------------------
# 3-7. 레이어 컨트롤(토글)
folium.LayerControl(collapsed=False, position='topright').add_to(m)

# ----------------------------------------------------------
# 4. 출력/저장
from IPython.display import display
display(m)
m.save('wri_complete_map_with_search.html')
